# Pikachu Robust RL - Colab T4
This notebook pins both repositories, verifies the production engine, resumes only verified episode-boundary checkpoints from Drive, and keeps validation separate from any sealed final set.

Private repository setup: add a GitHub token to Colab Secrets as `GITHUB_TOKEN` and enable notebook access. A fine-grained token with read-only Contents access is preferred; a classic PAT requires the `repo` scope. The token is passed through a temporary Git HTTP header and is never written to the clone URL or Git config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_URL   = 'https://github.com/jimin326/skku_pikachu.git'
PROJECT_ROOT  = '/content/skku_pikachu'
PROJECT_REF   = 'robust-rl-colab'
GAME_URL      = 'https://github.com/SKKU-x-HYU-SW-Competition/leonyi-volleyball.git'
GAME_COMMIT   = '1f3cecb90aca174ffc42ac6be4c384cc725d9e91'

DRIVE         = '/content/drive/MyDrive/pikachu_rl'
TRAINING_SEED = 20260905

# Which stage to run.  Each is a separate, resumable step; run them in order and
# stop as soon as a pre-registered gate says stop.  See bot-dev/rl/ABLATION_PLAN.md.
STAGE = 'A0'   # 'A0' | 'A1' | 'B'  | 'C'
DEVICE = 'cuda'  # set to 'cpu' only for a local dry run

# --- A0 / B: existing behaviour-cloning artifacts (never overwritten) ---
BC_DATASET = f'{DRIVE}/bc/v4_500k.jsonl'
BC_MODEL   = f'{DRIVE}/bc/v4_500k_ff.pt'

# --- A1: sweep the checkpoints of the already-finished BC+PPO run ---
SWEEP_SOURCE = f'{DRIVE}/bc_ppo_seed20260903/checkpoints'
SWEEP_OUT    = f'{DRIVE}/sweeps/bc_ppo_seed20260903'

# --- B: DAgger rounds on the train split only ---
DAGGER_ROUNDS    = 2
DAGGER_DECISIONS = 200_000
DAGGER_DIR       = f'{DRIVE}/dagger/seed{TRAINING_SEED}'
# Written by stage B, read by stage C.  Without it a fresh Colab runtime running
# STAGE='C' would silently fall back to the original (weak) BC model.
DAGGER_POINTER   = f'{DAGGER_DIR}/final_model.json'

# --- C: BC/KL-anchored PPO with validation-based selection ---
EXPERIMENT_ID = f'anchor_ppo_seed{TRAINING_SEED}'
RUN_DIR       = f'/content/runs/{EXPERIMENT_ID}'
RECOVERY      = f'{DRIVE}/{EXPERIMENT_ID}'
# Which policy the KL term pulls towards.  These are DIFFERENT arms, not a
# formatting detail: anchoring on the original BC model drags the policy back
# towards the weak teacher-distribution clone that scored 25% on its own.
#   'dagger' - anchor on the final DAgger model (default; anchor == init)
#   'bc'     - anchor on the original v4 BC model (separate ablation arm)
ANCHOR_SOURCE = 'dagger'
ANCHOR_KL     = 0.5
PHASE_STEPS   = 100_000
TOTAL_STEPS   = 1_000_000
LEARNING_RATE = 5e-5            # lower than the 3e-4 that produced experiment C
ENTROPY_COEF  = 0.005
MAX_WALL_MIN  = 210             # leave headroom inside a Colab session

assert ANCHOR_SOURCE in ('dagger', 'bc')
print('stage', STAGE, '| anchor', ANCHOR_SOURCE, '| drive', DRIVE)

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
from google.colab import userdata
try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a read-only GITHUB_TOKEN in Colab Secrets and enable notebook access') from exc
if not github_token:
    raise RuntimeError('Colab Secret GITHUB_TOKEN is empty')
basic_auth = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
git_env = os.environ.copy()
git_env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'Authorization: Basic {basic_auth}')
project_path = Path(PROJECT_ROOT)
if not (project_path / '.git').is_dir():
    if project_path.exists() and any(project_path.iterdir()):
        raise RuntimeError(f'{project_path} exists but is not a Git checkout; restart the runtime or clear that directory')
    subprocess.run(['git','clone','--branch',PROJECT_REF,'--single-branch',PROJECT_URL,PROJECT_ROOT], check=True, env=git_env)
subprocess.run(['git','-C',PROJECT_ROOT,'fetch','origin',PROJECT_REF], check=True, env=git_env)
subprocess.run(['git','-C',PROJECT_ROOT,'checkout','-B',PROJECT_REF,'origin/'+PROJECT_REF], check=True, env=git_env)
github_token = basic_auth = git_env = None  # discard notebook references to credentials
game_root = Path('/content/leonyi-volleyball')
if not (game_root / '.git').is_dir():
    if game_root.exists() and any(game_root.iterdir()):
        raise RuntimeError(f'{game_root} exists but is not a Git checkout; restart the runtime or clear that directory')
    subprocess.run(['git','clone',GAME_URL,str(game_root)], check=True)
subprocess.run(['git','-C',str(game_root),'checkout',GAME_COMMIT], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-rl.txt'], check=True)
subprocess.run(['node','scripts/setup_rl_engine.mjs',str(game_root)], check=True)
print('Prepared', PROJECT_REF, 'at', subprocess.run(['git','rev-parse','HEAD'], cwd=PROJECT_ROOT, check=True, text=True, capture_output=True).stdout.strip())


In [ ]:
import subprocess, sys
checks = [
    ['node','bot-dev/rl/physics_clamp_smoke.mjs'],
    ['node','bot-dev/rl/production_differential.mjs','--game-root','/content/leonyi-volleyball'],
    ['node','bot-dev/rl/env_smoke.mjs'],
    [sys.executable,'bot-dev/rl/bridge_smoke.py'],
    [sys.executable,'bot-dev/rl/ppo_tests.py'],
    [sys.executable,'bot-dev/rl/anchor_tests.py'],
    [sys.executable,'bot-dev/rl/pipeline_tests.py'],
    [sys.executable,'bot-dev/rl/eval/test_schema.py'],
    [sys.executable,'bot-dev/rl/eval/test_stats.py'],
    ['node','bot-dev/rl/eval/paired_eval_smoke.mjs'],
]
for command in checks:
    print('RUN', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime before training'
print(torch.cuda.get_device_name(0))


## A0 - re-run the BC diagnostics on the real 500k dataset

Answers: is the teacher's action ambiguous given the 4-frame observation
(hidden-state aliasing), or is the supervised fit fine and the failure in the
rollout (covariate shift)?  A 200k replication in this repository measured a
0.0065% label-conflict rate and 91.2% episode-held-out accuracy, i.e. **not**
aliasing.  Confirm that on the real dataset before choosing an algorithm.

In [ ]:
import subprocess, sys, json
from pathlib import Path
if STAGE == 'A0':
    out = f'{DRIVE}/diagnostics/bc_500k_diag.json'
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    assert Path(BC_DATASET).is_file(), f'missing {BC_DATASET}'
    subprocess.run([sys.executable, 'bot-dev/rl/bc_diagnostics.py', BC_DATASET,
                    '--model', BC_MODEL, '--train-epochs', '10',
                    '--device', 'cuda', '--output', out], cwd=PROJECT_ROOT, check=True)
    report = json.loads(Path(out).read_text())
    print(json.dumps({
        'majorityHeldout': report['majorityBaseline']['heldoutAccuracy'],
        'retrainHeldout': report.get('heldoutRetrain', {}).get('finalHeldoutAccuracy'),
        'aliasing': [{k: a[k] for k in ('frames','conflictSampleFraction','accuracyCeiling','conditionalEntropyBits')} for a in report['aliasing']],
    }, indent=2))
    print('conflict fraction near 0 -> aliasing is NOT the bottleneck; do not promote the recurrent arm.')

## A1 - checkpoint sweep of the finished BC+PPO run

No training.  Evaluates every saved checkpoint on the same paired validation set
and reports where the collapse began, plus KL / argmax drift against the BC
anchor.  Resumable: re-running skips checkpoints already in `sweep.jsonl`.

In [ ]:
import subprocess, sys, json
from pathlib import Path
if STAGE == 'A1':
    assert Path(SWEEP_SOURCE).is_dir(), f'missing {SWEEP_SOURCE}'
    command = [sys.executable, 'bot-dev/rl/checkpoint_sweep.py', SWEEP_SOURCE,
               '--output-dir', SWEEP_OUT, '--anchor', BC_MODEL,
               '--probe-dataset', BC_DATASET, '--probe-limit', '20000',
               '--reference', BC_MODEL, '--max-checkpoints', '12', '--runtime']
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    summary = json.loads((Path(SWEEP_OUT) / 'sweep_summary.json').read_text())
    print(json.dumps(summary['onset'], indent=2))
    for row in summary['table']:
        print(row)
    print('best:', summary['best'])

## B - DAgger rounds (train split only)

`teacher_shadow.mjs` runs the current policy in real matches and records the
frozen v4 teacher's action on every **learner-visited** state.  The dataset is
refused unless the split is `train`, so validation opponents and seeds cannot
leak into training data.

Measured in this repository at 200k + 2x100k on CPU: non-benchmark paired delta
-0.875 -> -0.281, match win rate 25.00% -> 64.58%.  Self-destruction stayed at
32.6%, so this alone does not pass the gates.

## Shared validation helper

Stage B uses the *same* export -> paired evaluation -> stats -> runtime -> gates
path as stage C, so a DAgger round can never be accepted without being measured
on the held-out split.

In [ ]:
# Shared validation helper: export -> paired eval -> stats -> runtime -> gates.
# Used after every DAgger round so a round is never accepted unmeasured.
import json, subprocess, sys
from pathlib import Path

def validate_model(model_path, out_dir, tag):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    bot   = out_dir / f'{tag}.js'
    raw   = out_dir / 'validation.jsonl'
    stats = out_dir / 'validation_stats.json'
    rt    = out_dir / 'runtime.json'
    gates = out_dir / 'gates.json'
    shadow = out_dir / 'shadow_validation.json'
    if not gates.is_file():
        subprocess.run([sys.executable,'bot-dev/rl/export_policy.py',str(model_path),str(bot)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        subprocess.run([sys.executable,'bot-dev/rl/export_policy_test.py',str(model_path),str(bot)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        subprocess.run(['node','bot-dev/rl/export_env_smoke.mjs',str(bot)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        subprocess.run(['node','bot-dev/rl/eval/paired_eval.mjs',f'--candidate={bot}',f'--output={raw}'], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        subprocess.run([sys.executable,'bot-dev/rl/eval/stats.py',str(raw),'--output',str(stats)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        with rt.open('w') as sink:
            subprocess.run(['node','--expose-gc','bot-dev/rl/eval/runtime_bench.mjs',f'--candidate={bot}'], cwd=PROJECT_ROOT, check=True, stdout=sink)
        subprocess.run([sys.executable,'bot-dev/rl/eval/gates.py',str(stats),'--rows',str(raw),'--runtime',str(rt),'--output',str(gates)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        subprocess.run(['node','bot-dev/rl/eval/teacher_shadow.mjs',f'--candidate={bot}',f'--output={shadow}'], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
    result = json.loads(gates.read_text())
    result['teacherDisagreement'] = json.loads(shadow.read_text())['disagreement']['overall'] if shadow.is_file() else None
    result['candidateJs'] = str(bot)
    return result
print('validate_model ready')

In [ ]:
# DAgger rounds.  Every round is validated on the held-out split immediately,
# and the go/no-go rule below decides whether another round is justified.
if STAGE == 'B':
    Path(DAGGER_DIR).mkdir(parents=True, exist_ok=True)
    current_model, history = Path(BC_MODEL), []
    sources = [(0, Path(BC_DATASET))]

    baseline = validate_model(current_model, f'{DAGGER_DIR}/validation_r0', 'bc_r0')
    history.append({'round': 0, **baseline['summary'], 'selectable': baseline['selectable'],
                    'submission': baseline['submission'], 'disagreement': baseline['teacherDisagreement']})
    print('round 0:', json.dumps(history[-1]))

    for rnd in range(1, DAGGER_ROUNDS + 1):
        bot = Path(DAGGER_DIR) / f'policy_r{rnd - 1}.js'
        labels = Path(DAGGER_DIR) / f'round{rnd}.jsonl'
        model_out = Path(DAGGER_DIR) / f'dagger_r{rnd}.pt'
        if not bot.is_file():
            subprocess.run([sys.executable,'bot-dev/rl/export_policy.py',str(current_model),str(bot)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
            subprocess.run([sys.executable,'bot-dev/rl/export_policy_test.py',str(current_model),str(bot)], cwd=PROJECT_ROOT, check=True, stdout=subprocess.DEVNULL)
        if not labels.is_file():
            subprocess.run(['node','bot-dev/rl/eval/teacher_shadow.mjs',f'--candidate={bot}',
                            '--split=bot-dev/rl/eval/splits/train.json','--beta=0',
                            f'--decisions={DAGGER_DECISIONS}',f'--mix-seed={TRAINING_SEED + rnd}',
                            f'--dataset-output={labels}',
                            f'--output={DAGGER_DIR}/shadow_train_r{rnd}.json'], cwd=PROJECT_ROOT, check=True)
        sources.append((rnd * 10_000_000, labels))
        if not model_out.is_file():
            aggregate = Path(DAGGER_DIR) / f'aggregate_r{rnd}.jsonl'
            total = 0
            with aggregate.open('w', encoding='utf-8') as sink:
                for offset, source in sources:
                    for line in source.open(encoding='utf-8'):
                        if not line.strip(): continue
                        row = json.loads(line)
                        sink.write(json.dumps({'observation': row['observation'], 'action': row['action'],
                                               'episodeStart': row['episodeStart'], 'episodeId': row['episodeId'] + offset,
                                               'seed': row['seed'], 'side': row['side'], 'opponentId': row['opponentId']}) + '\n')
                        total += 1
            meta = json.loads(Path(str(BC_DATASET) + '.meta.json').read_text())
            meta.update({'decisions': total, 'collection': f'bc + dagger rounds 1..{rnd}', 'daggerRounds': rnd})
            Path(str(aggregate) + '.meta.json').write_text(json.dumps(meta, indent=2) + '\n')
            subprocess.run([sys.executable,'bot-dev/rl/bc_pretrain.py',str(aggregate),str(model_out),
                            '--epochs=10',f'--device={DEVICE}',f'--seed={TRAINING_SEED}'], cwd=PROJECT_ROOT, check=True)
        current_model = model_out

        # Validate THIS round before deciding to run another one.
        result = validate_model(current_model, f'{DAGGER_DIR}/validation_r{rnd}', f'dagger_r{rnd}')
        history.append({'round': rnd, **result['summary'], 'selectable': result['selectable'],
                        'submission': result['submission'], 'disagreement': result['teacherDisagreement']})
        print('round', rnd, ':', json.dumps(history[-1]))

        # Pre-registered go/no-go: self-destruction must not rise two rounds in a row.
        sd = [h['selfDestruction'] for h in history]
        if len(sd) >= 3 and sd[-1] > sd[-2] > sd[-3]:
            print('STOP: self-destruction rose for two consecutive rounds', sd[-3:])
            break

    Path(DAGGER_POINTER).write_text(json.dumps({
        'model': str(current_model), 'rounds': len(history) - 1,
        'history': history, 'seed': TRAINING_SEED}, indent=2))
    print(json.dumps(history, indent=2))
    print('pointer written:', DAGGER_POINTER)

    # Pre-registered go/no-go for entering stage C (ABLATION_PLAN.md section 5).
    final = history[-1]
    checks = {
        'selfDestruction < 0.25': final['selfDestruction'] < 0.25,
        'primary CI lower >= -0.20': json.loads((Path(f"{DAGGER_DIR}/validation_r{final['round']}") / 'gates.json').read_text())['gates']['primary_ci_lower']['value'] >= -0.20,
        'disagreement decreased vs round 0': (final['disagreement'] is not None and history[0]['disagreement'] is not None
                                              and final['disagreement'] < history[0]['disagreement']),
    }
    print(json.dumps(checks, indent=2))
    held = [name for name, ok in checks.items() if ok]
    if not held:
        print('NO-GO: none of the pre-registered conditions held. Stop the pure-policy path, report the')
        print('       experiment, and consider the v4-policy + rally-level RL selector hybrid instead.')
    elif len(held) < len(checks):
        print(f'WEAK GO ({len(held)}/{len(checks)}): only {held} held. Stage C is worth at most 2-3 phases;')
        print('       stop it as soon as self-destruction rises twice or no checkpoint is selectable.')
    else:
        print('GO: all conditions held. Proceed to STAGE = "C".')

## C - BC/KL-anchored PPO with held-out validation, selection and rollback

`train_with_validation.py` trains one phase, validates on the held-out split,
applies the pre-registered gates in `eval/gates.py`, keeps the best checkpoint,
and rolls back to it (with a halved learning rate) when two phases in a row do
not improve.  Everything needed to continue lives in `RECOVERY` on Drive, so a
disconnect costs at most one phase.  Re-run this cell to resume.

The final answer is `best.json`, never `latest.pt`.

In [ ]:
# BC/KL-anchored PPO.  Resumable; re-run this cell to continue after a disconnect.
if STAGE == 'C':
    pointer = Path(DAGGER_POINTER)
    if pointer.is_file():
        info = json.loads(pointer.read_text())
        initial = info['model']
        print(f"initialising from the stage-B DAgger model ({info['rounds']} rounds): {initial}")
    else:
        initial = BC_MODEL
        print('WARNING: no DAgger pointer found; initialising from the original BC model.')
        print('         Run STAGE="B" first unless you deliberately want the BC-init arm.')
    anchor = initial if ANCHOR_SOURCE == 'dagger' else BC_MODEL
    print('anchor arm:', ANCHOR_SOURCE, '->', anchor)
    experiment = f'{EXPERIMENT_ID}_anchor-{ANCHOR_SOURCE}'
    run_dir  = f'/content/runs/{experiment}'
    recovery = f'{DRIVE}/{experiment}'
    command = [sys.executable, 'bot-dev/rl/train_with_validation.py',
               '--experiment-id', experiment, '--run-dir', run_dir, '--recovery-dir', recovery,
               '--initial-model', str(initial), '--anchor-model', str(anchor),
               '--anchor-kl-coef', str(ANCHOR_KL), '--policy-freeze-updates', '10',
               '--learning-rate', str(LEARNING_RATE), '--entropy-coef', str(ENTROPY_COEF),
               '--phase-steps', str(PHASE_STEPS), '--total-steps', str(TOTAL_STEPS),
               '--patience', '2', '--max-rollbacks', '2', '--max-unselectable-phases', '3',
               '--workers', '4', '--envs-per-worker', '4', '--seed', str(TRAINING_SEED),
               '--device', 'auto', '--save-every-minutes', '20',
               '--max-wall-minutes', str(MAX_WALL_MIN),
               '--probe-dataset', BC_DATASET, '--probe-limit', '20000', '--runtime']
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    best_path = Path(run_dir) / 'best.json'
    if best_path.is_file():
        print(json.dumps(json.loads(best_path.read_text()), indent=2))
    else:
        print('No selectable checkpoint was produced; see state.json for the stop reason.')
    RUN_DIR = run_dir

## Final gate check

Validation only.  Do not put sealed-final seeds in this notebook, and do not run
extra training seeds until `submission` is true.

In [ ]:
import json, shutil
from pathlib import Path
candidates = []
for name, path in (('sweep', Path(SWEEP_OUT) / 'sweep.jsonl'),
                   ('training', Path(globals().get('RUN_DIR', '/content/runs/none')) / 'validation_log.jsonl')):
    if path.is_file():
        candidates += [(name, json.loads(line)) for line in path.read_text().splitlines() if line.strip()]
for gates_path in sorted(Path(DAGGER_DIR).glob('validation_r*/gates.json')):
    gates = json.loads(gates_path.read_text())
    candidates.append(('dagger', {'step': 0, 'primary': {'estimate': gates['summary']['primaryEstimate']},
                                  'benchmark': {'estimate': gates['summary']['benchmarkEstimate']},
                                  'selfDestruction': {'rateAmongLosses': gates['summary']['selfDestruction']},
                                  'runtime': gates['gates']['runtime'], 'selectable': gates['selectable'],
                                  'submission': gates['submission'], 'selectionKey': gates['selectionKey'],
                                  'workDir': str(gates_path.parent)}))
if not candidates:
    print('nothing evaluated yet')
else:
    best_source, best = max(candidates, key=lambda item: tuple(item[1]['selectionKey']))
    print(json.dumps({'source': best_source, 'step': best['step'], 'primary': best['primary'],
                      'benchmark': best['benchmark'], 'selfDestruction': best['selfDestruction'],
                      'runtime': best['runtime'], 'selectable': best['selectable'],
                      'submission': best['submission'], 'workDir': best['workDir']}, indent=2))
    if best['submission']:
        print('All pre-registered gates pass.  Next step is 3 independent training seeds, then the sealed final.')
        print('Candidate JavaScript:', Path(best['workDir']) / 'candidate.js')
    else:
        print('Gates FAILED - report the experiment only.  Do not create a submission model in src/code-here,')
        print('do not run additional seeds, and do not open the sealed final set.')